# Example - Nonlinear Model with Feedback Control

In this example, we construct a nonlinear kinetic model of a drone in 2D space (i.e. top-down x, y position).  Then we add linear models to the inputs to reflect the dynamic response of the motors.  Finally, we add PI (proportional-integral) and P (proportional) controllers in cascaded feedback loops to drive the drone towards a target position.

<img src="images/drone_control_system_diagram.png" width="80%" style="display: block; margin: auto">

## Constructing Nonlinear State-Space Models

For continuous-time models, these consist of two functions, `f` and `h`, in the following form:

$$
\begin{aligned}
\dot{x}(t) = f(t, x(t), u(t)) \\
y(t) = h(t, x(t), u(t))
\end{aligned}
$$

where $x(t)$ is the state vector, $u(t)$ is an input vector and $\dot{x}(t)$ is the time derivative of $x$.

Note that the time variable, `t`, is not often used in the model equations, but the functions must have it as an argument in any case.

To construct a state-space model, start by defining $f$ and $h$ as CasADi functions in the following form:

```none
Function(f:(t,x[n],u[nu])->(rhs[n]) SXFunction)
Function(h:(t,x[n],u[nu])->(y[ny]) SXFunction)
```

where `n` is the length of the state vector, `nu` is the number of inputs, and `ny` is the number of outputs.

## Drone Body Dynamics

This is a free body kinetics model with a drag force proportional to velocity squared.

In [ ]:
import casadi as cas

# Time variable
t = cas.SX.sym('t')

# States
p = cas.SX.sym('p', 2)  # position vector (pos_x, pos_y)
v = cas.SX.sym('v', 2)  # velocity vector (vel_x, vel_y)
x = cas.vertcat(p, v)  # state vector

# Input
u = cas.SX.sym('u', 2)  # thrust force vector (thrust_x, thrust_y)

# Constants
m = cas.DM(0.1)  # mass
c = cas.DM(5.0e-4)  # drag coefficient
EPS = cas.DM(1e-10)  # constant to avoid numerical issues at v=0

speed = cas.sqrt(cas.sumsqr(v) + EPS)

# Righthand side of the ODE
rhs = cas.vertcat(
    v,
    (u - c * speed * v) / m
)

# Construct ODE function
f = cas.Function('f', [t, x, u], [rhs], ['t', 'x', 'u'], ['rhs'])

# Construct output function
y = x
h = cas.Function('h', [t, x, u], [y], ['t', 'x', 'u'], ['y'])

# System dimensions
n = 4  # x.shape[0]
nu = 2  # u.shape[0]
ny = 4  # y.shape[0]

print(f)
print(h)

Pass these functions to the state space model constructor and provide variable names.

In [ ]:
from cas_models.continuous_time.models import StateSpaceModelCT

# Variable names
input_names = ['thrust_x', 'thrust_y']
state_names = ['pos_x', 'pos_y', 'vel_x', 'vel_y']
output_names = state_names

model = StateSpaceModelCT(
    f, h,
    n, nu, ny,
    name='body',
    input_names=input_names,
    state_names=state_names,
    output_names=output_names
)
model.describe()

## Connect Motor Models

Casadi-models provides common pre-defined linear system models that you can easily connect to your nonlinear model.

For example, here is a first order, single-input, single output (SISO) linear model.

In [ ]:
from cas_models.continuous_time.models import (
    SSModelCTLinearFOSISO, SSModelCTLinearFONoGainSISO
)

# First order linear single-input, single-output (SISO) system
motor_model = SSModelCTLinearFOSISO(K=2.5, T1=0.2, name='motor')
motor_model.describe()

Connect models together.

In [ ]:
from cas_models.transformations import (
    connect_systems_in_series, connect_systems_in_parallel
)

# Combine two identical motor models to create a 2-input, 2-output system
motor_models = connect_systems_in_parallel(
    [motor_model] * 2, model_class=StateSpaceModelCT, name="motors"
)

# Connect both sub-systems together
sys = connect_systems_in_series(
    [motor_models, model], model_class=StateSpaceModelCT, name="drone"
)
sys.describe()

Note: When unique variable names for the combined system are not specified explicitly, they are created automatically.  For example `motor1_x` and `motor1_u` above.

In [ ]:
# Series connections can also be made with the '*' operator
# Note: The order of multiplication is not the same as
# connect_systems_in_series above
sys = model * motor_models
sys.name = "drone"
sys.describe()

## Discretization

Continuous-time model can be converted to an equivalent (numerically approximate) discrete-time model using CasADi's built in integration tools.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cas_models.discrete_time.models import StateSpaceModelDTFromCT

# Convert to discrete-time model with time interval dt
dt = 0.1
sys_dt = StateSpaceModelDTFromCT(sys, dt)
sys_dt.describe()

Note that the state transition function, `F` and output function, `H`, for discrete-time system models have different names to avoid confusion with `f` and `h` for continuous time systems.  The model also has an additional parameter `dt` for the time interval. The states, inputs and outputs are also named differently (`xk`, `uk`, and `yk`).

## Simulation

The function `make_n_step_simulation_function_from_model` constructs a simulation function for a finite time trajectory rollout with the given discrete-time system model.

In [ ]:
from cas_models.discrete_time.simulate import (
    make_n_step_simulation_function_from_model
)

# Number of time-steps to simulate
nT = 200

# Create CasADi simulation function
simulate = make_n_step_simulation_function_from_model(sys_dt, nT)
print(simulate)

Note that the input data sequence, `U`, is one time step shorter than the time sequence, `t_eval`, which includes the final time instant, `nT + 1`.

This function can be used to simulate trajectories of the system given an initial state and input sequence.

In [ ]:
# Simulation time
t = dt * np.arange(nT + 1)
t_in = t[:-1]  # excludes final time value

# Step inputs
U = np.zeros((nT, sys_dt.nu))
U[t_in >= 1, 0] = 1.0
U[t_in >= 3, 1] = 1.0

# Initial condition
x0 = np.zeros(sys_dt.n)
X, Y = simulate(t, U, x0)

X = np.array(X)
Y = np.array(Y)

assert X.shape == (nT + 1, sys_dt.n)  # states
assert Y.shape == (nT + 1, sys_dt.ny)  # outputs

In [ ]:
# Make time-series plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5, 3.5), sharex=True)

ax1.plot(t, Y[:, 2], label=sys.output_names[2])
ax1.plot(t, Y[:, 3], label=sys.output_names[3])
ax1.set_ylabel("y")
ax1.legend()
ax1.set_title("System Outputs - Velocity")
ax1.grid(False)

ax2.step(t_in, U[:, 0], where="post", label=sys.input_names[0])
ax2.step(t_in, U[:, 1], where="post", label=sys.input_names[1])
ax2.set_xlabel("t")
ax2.set_ylabel("u")
ax2.legend()
ax2.set_title("System Inputs")
ax2.grid(False)

plt.tight_layout()
plt.show()

## Design PI Controller for Inner Loop Velocity Control

In [ ]:
# Steady state gain
Y[-1, 2:] / U[-1, 0:2]

In [ ]:
# 1st order time constant
t_632 = np.argmin((Y[:, 2] - 0.632*59.42) ** 2)
t[t_632] - 1

In [ ]:
# 1st order time constant
t_632 = np.argmin((Y[:, 3] - 0.632*59.42) ** 2)
t[t_632] - 3

In [ ]:
# PI controller tuning based on step responses
Kp = 59.0
T1 = 2.6
ctrls_inner_Kc = 1 / Kp
ctrls_inner_Ti = T1 * 2  # a bit conservative

## Construct the Inner Closed-Loop System

In [ ]:
from cas_models.continuous_time.regulators import SSModelCTPIInt

# Same PI controller (interactive form) for both directions
# so Kc and Ti are shared parameters
pi_ctrl_inner = SSModelCTPIInt(name="PI")
ctrls_inner = connect_systems_in_parallel(
    [pi_ctrl_inner] * 2, model_class=StateSpaceModelCT, name="ctrls"
)
ctrls_inner.describe()

In [ ]:
from cas_models.transformations import connect_systems

# Connect the controllers and system together
# with the control action as outputs.
sys_cl_inner = connect_systems(
    [ctrls_inner, sys],
    connections={
        'ctrls_PI1_e': {'ref_vel_x': 1.0, 'drone_vel_x': -1.0},
        'ctrls_PI2_e': {'ref_vel_y': 1.0, 'drone_vel_y': -1.0},
        'drone_motor1_u': 'ctrls_PI1_u',
        'drone_motor2_u': 'ctrls_PI2_u',
    },
    model_class=StateSpaceModelCT,
    input_names=['ref_vel_x', 'ref_vel_y'],
    output_names=[
        'ctrls_PI1_u', 'ctrls_PI2_u',
        'drone_pos_x', 'drone_pos_y',
        'drone_vel_x', 'drone_vel_y'
    ],
    verbose_names=True,
    name="inner_loop"
)
sys_cl_inner.describe()

## Simulating the Inner Feedback Loop

In [ ]:
# Convert to discrete-time model and create simulation function
sys_cl_inner_dt = StateSpaceModelDTFromCT(sys_cl_inner, dt)
simulate_cl_inner = make_n_step_simulation_function_from_model(
    sys_cl_inner_dt, nT
)
simulate_cl_inner

In [ ]:
ref_vel_x = np.zeros_like(t)
ref_vel_x[10:] = 1.0
ref_vel_y = np.zeros_like(t)
ref_vel_y[50:] = 1.0

R_vel_full = np.column_stack([ref_vel_x, ref_vel_y])
R_vel_in = R_vel_full[:-1]

x0 = np.zeros(sys_cl_inner_dt.n)
assert t.shape == (nT + 1,)
assert R_vel_in.shape == (nT, 2)
X_cl_inner, Y_cl_inner = simulate_cl_inner(
    t, R_vel_in, x0, ctrls_inner_Kc, ctrls_inner_Ti
)
X_cl_inner = np.array(X_cl_inner)
Y_cl_inner = np.array(Y_cl_inner)

assert X_cl_inner.shape == (nT + 1, sys_cl_inner_dt.n)
assert Y_cl_inner.shape == (nT + 1, sys_cl_inner_dt.ny)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 2.5))

ax.step(t_in, R_vel_in[:, 0], where="post", linestyle="--", label="ref_vel_x")
ax.step(t_in, R_vel_in[:, 1], where="post", linestyle="--", label="ref_vel_y")
ax.plot(t, Y_cl_inner[:, 4], "C0", label="vel_x")
ax.plot(t, Y_cl_inner[:, 5], "C1", label="vel_y")
ax.set_xlabel("Time (t)")
ax.set_ylabel("vel")
ax.legend()
ax.grid(False)
ax.set_title("Inner Closed Loop Response")

plt.tight_layout()
plt.show()

## Design P Controller for Outer Loop Position Control

In [ ]:
# P controller tuning - must be slower than inner loop to ensure stability
ctrls_outer_Kc = 0.1

## Construct the Outer Feedback Loop

In [ ]:
from cas_models.continuous_time.regulators import SSModelCTP

# Same P controller for both directions so Kc is a shared parameter
p_ctrl_outer = SSModelCTP(name="P")
ctrls_outer = connect_systems_in_parallel(
    [p_ctrl_outer] * 2,
    model_class=StateSpaceModelCT,
    name="ctrls_outer"
)
ctrls_outer.describe()

In [ ]:
# Add setpoint filters to slow down the response to setpoint changes
spf = SSModelCTLinearFONoGainSISO(name="spf")
sp_filters = connect_systems_in_parallel(
    [spf] * 2,
    model_class=StateSpaceModelCT,
    name="spfs"
)
sp_filters.describe()

In [ ]:
# Connect the controllers and system together
# with the control action as outputs.
sys_cl_outer = connect_systems(
    [sp_filters, ctrls_outer, sys_cl_inner],
    connections={
        'spfs_spf1_u': 'ref_pos_x',
        'spfs_spf2_u': 'ref_pos_y',
        'ctrls_outer_P1_e': {
            'spfs_spf1_y': 1.0,
            'inner_loop_drone_pos_x': -1.0
        },
        'ctrls_outer_P2_e': {
            'spfs_spf2_y': 1.0,
            'inner_loop_drone_pos_y': -1.0
        },
        'inner_loop_ref_vel_x': 'ctrls_outer_P1_u',
        'inner_loop_ref_vel_y': 'ctrls_outer_P2_u',
    },
    model_class=StateSpaceModelCT,
    input_names=['ref_pos_x', 'ref_pos_y'],
    output_names=[
        'ctrls_outer_P1_u', 'ctrls_outer_P2_u',
        'inner_loop_ctrls_PI1_u', 'inner_loop_ctrls_PI2_u',
        'inner_loop_drone_pos_x', 'inner_loop_drone_pos_y',
        'inner_loop_drone_vel_x', 'inner_loop_drone_vel_y'
    ],
    verbose_names=True
)
sys_cl_outer.describe()

## Simulate the Complete Closed-Loop System

In [ ]:
# Convert to discrete-time model and create simulation function
nT = 1000
sys_cl_outer_dt = StateSpaceModelDTFromCT(sys_cl_outer, dt)
simulate_cl_outer = make_n_step_simulation_function_from_model(
    sys_cl_outer_dt, nT
)
simulate_cl_outer

In [ ]:
# Simulation time
t = dt * np.arange(nT + 1)
t_in = t[:-1]  # excludes final time value

# Step inputs
ref_pos_x = np.zeros_like(t)
ref_pos_x[10:] = 1.0
ref_pos_y = np.zeros_like(t)
ref_pos_y[200:] = 1.0

ref_pos_x[400:] = 0.5
ref_pos_y[400:] = 0.5

R_pos_full = np.column_stack([ref_pos_x, ref_pos_y])
R_pos_in = R_pos_full[:-1]

# Setpoint filter time constant
spfs_T1 = 10.0

x0 = np.zeros(sys_cl_outer_dt.n)
assert t.shape == (nT + 1,)
assert R_pos_in.shape == (nT, sys_cl_outer_dt.nu)

X_cl_outer, Y_cl_outer = simulate_cl_outer(
    t, R_pos_in, x0, spfs_T1,
    ctrls_outer_Kc, ctrls_inner_Kc, ctrls_inner_Ti
)
X_cl_outer = np.array(X_cl_outer)
Y_cl_outer = np.array(Y_cl_outer)

assert X_cl_outer.shape == (nT + 1, sys_cl_outer_dt.n)
assert Y_cl_outer.shape == (nT + 1, sys_cl_outer_dt.ny)

In [ ]:
pos_x = Y_cl_outer[:, 4]
pos_y = Y_cl_outer[:, 5]
vel_x = Y_cl_outer[:, 6]
vel_y = Y_cl_outer[:, 7]
ref_vel_x = Y_cl_outer[:, 0]
ref_vel_y = Y_cl_outer[:, 1]

fig = plt.figure(figsize=(5, 7.5), layout='constrained')
gs = fig.add_gridspec(3, 1, height_ratios=[4, 1, 1])
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])
ax3 = fig.add_subplot(gs[2], sharex=ax2)

# Top: 2D position trajectory vs reference waypoints
ax1.plot(pos_x, pos_y, "C0", label="trajectory")
ax1.plot(pos_x[0], pos_y[0], "o", color="C0", markersize=5)

ref_xy = np.column_stack([R_pos_in[:, 0], R_pos_in[:, 1]])
_, idx = np.unique(ref_xy, axis=0, return_index=True)
ref_waypoints = ref_xy[np.sort(idx)]
ax1.plot(
    ref_waypoints[:, 0], ref_waypoints[:, 1],
    "k--o", markersize=6, label="setpoint"
)
ax1.set_aspect('equal', adjustable='datalim')
ax1.set_xlabel("pos_x")
ax1.set_ylabel("pos_y")
ax1.set_title("Position Trajectory")
ax1.legend()
ax1.grid(True)

# Middle: position time series
ax2.step(t_in, R_pos_in[:, 0], where="post", linestyle="--", label="ref_pos_x")
ax2.step(t_in, R_pos_in[:, 1], where="post", linestyle="--", label="ref_pos_y")
ax2.plot(t, pos_x, "C0", label="pos_x")
ax2.plot(t, pos_y, "C1", label="pos_y")
ax2.set_ylabel("pos")
ax2.legend(fontsize="x-small")
ax2.grid(True)
ax2.set_title("Position vs. Time")

# Bottom: velocity time series
ax3.plot(t, ref_vel_x, "C0", linestyle="--", label="ref_vel_x")
ax3.plot(t, ref_vel_y, "C1", linestyle="--", label="ref_vel_y")
ax3.plot(t, vel_x, "C0", label="vel_x")
ax3.plot(t, vel_y, "C1", label="vel_y")
ax3.set_xlabel("Time (t)")
ax3.set_ylabel("vel")
ax3.legend(fontsize="x-small")
ax3.grid(True)
ax3.set_title("Velocity vs. Time")
plt.show()

## Compile Simulation Results and Save to File

In [ ]:
import pandas as pd

inputs = pd.DataFrame(R_pos_in, columns=sys_cl_outer_dt.input_names)
outputs = pd.DataFrame(Y_cl_outer, columns=sys_cl_outer_dt.output_names)
sim_results = pd.concat([inputs, outputs], axis=1)

sim_results.head()

In [ ]:
import os

results_dir = "results"

os.makedirs(results_dir, exist_ok=True)
sim_results.to_csv(
    os.path.join(results_dir, "drone_sim_results.csv"),
    index=False
)